# 18 · Audio Prediction Module Verification (predict_audio)
End-to-end verification of `src.models.audio.predict_audio`: schema conformity on held-out fold 10, exact prediction agreement against precomputed features and intensity scores, baseline accuracy check (0.7818), error handling for missing files, latency benchmark, and artifact publication to `models/audio_model.joblib`.

In [1]:
# Cell 1 – interpreter setup, imports, and input SHA256 asserts
import sys
from pathlib import Path
import hashlib
import time
import shutil

print('sys.executable:', sys.executable)
REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print('REPO_ROOT     :', REPO_ROOT)

import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import accuracy_score

from src.models.audio import predict_audio

print('Successfully imported predict_audio and libraries')

def compute_sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while chunk := f.read(65536):
            h.update(chunk)
    return h.hexdigest()

EXPECTED_HASHES = {
    REPO_ROOT / 'models/svm_mixed_candidate.joblib': '72543a986a3178f952f7a935ab7054ad44cfc1e52dbcae30a2756fee95dbb03d',
    REPO_ROOT / 'models/intensity_bounds.joblib'    : '22b0cdb61eb19769edb41f1e88049f7625e2b9b089cec9788f165ff8e4155340',
    REPO_ROOT / 'data/processed/audio_features.npy' : 'ed16a16f2a9e5c90aa61e925b461e0bce053c4654f1cc22a0812264630b1c73a',
    REPO_ROOT / 'data/processed/labels.csv'         : '8b4194ff7b2f47048c0e70f5ddfcc9c19d1994079405584dc8e86b9180d5f854',
    REPO_ROOT / 'data/processed/split_indices.npz'  : '2c0cdd397d4d9a7c3decf630a214cae2f55e7bd8c0ead5693a79d36b4cc8b10b',
    REPO_ROOT / 'data/processed/audio_intensity.npy': '1277608d8edcc60b5d5bb85a2f04988600c5be3264dc152b0ba8edc4de33f811',
}

print('\n=== Verifying SHA256 of Required Artifacts ===')
for path, expected_sha in EXPECTED_HASHES.items():
    actual_sha = compute_sha256(path)
    rel_path = path.relative_to(REPO_ROOT)
    print(f'{str(rel_path):42s}: {actual_sha}')
    assert actual_sha == expected_sha, f'SHA256 mismatch for {rel_path}: expected {expected_sha}, got {actual_sha}'

print('All 6 input artifact SHA256 hashes verified successfully ✓')

sys.executable: C:\Users\LENOVO\miniconda3\envs\citypulse\python.exe
REPO_ROOT     : C:\Users\LENOVO\Desktop\CityPulse


Successfully imported predict_audio and libraries

=== Verifying SHA256 of Required Artifacts ===
models\svm_mixed_candidate.joblib         : 72543a986a3178f952f7a935ab7054ad44cfc1e52dbcae30a2756fee95dbb03d
models\intensity_bounds.joblib            : 22b0cdb61eb19769edb41f1e88049f7625e2b9b089cec9788f165ff8e4155340
data\processed\audio_features.npy         : ed16a16f2a9e5c90aa61e925b461e0bce053c4654f1cc22a0812264630b1c73a
data\processed\labels.csv                 : 8b4194ff7b2f47048c0e70f5ddfcc9c19d1994079405584dc8e86b9180d5f854
data\processed\split_indices.npz          : 2c0cdd397d4d9a7c3decf630a214cae2f55e7bd8c0ead5693a79d36b4cc8b10b
data\processed\audio_intensity.npy        : 1277608d8edcc60b5d5bb85a2f04988600c5be3264dc152b0ba8edc4de33f811
All 6 input artifact SHA256 hashes verified successfully ✓


In [2]:
# Cell 2 – run predict_audio on all 605 fold-10 clips and assert schema
labels = pd.read_csv(REPO_ROOT / 'data/processed/labels.csv')
with np.load(REPO_ROOT / 'data/processed/split_indices.npz') as sp:
    test_idx = sp['test_idx']

assert len(test_idx) == 605, f'Expected 605 fold-10 clips, got {len(test_idx)}'

TARGET_CLASSES = {'construction', 'hvac', 'music_loudspeaker', 'siren', 'traffic_horn'}
REQUIRED_KEYS = {'type', 'confidence', 'relative_intensity'}

test_outputs = []
t0 = time.perf_counter()

for i, idx in enumerate(test_idx):
    rel_path = labels.loc[idx, 'path']
    full_path = REPO_ROOT / rel_path
    
    out = predict_audio(full_path)
    
    # Assert output schema and valid ranges
    assert isinstance(out, dict), f'Clip {idx}: output is not a dict'
    assert set(out.keys()) == REQUIRED_KEYS, f'Clip {idx}: keys {set(out.keys())} != {REQUIRED_KEYS}'
    assert out['type'] in TARGET_CLASSES, f'Clip {idx}: unexpected type {out["type"]}'
    assert isinstance(out['confidence'], float), f'Clip {idx}: confidence is not float'
    assert 0.0 <= out['confidence'] <= 1.0, f'Clip {idx}: confidence {out["confidence"]} out of [0, 1]'
    assert isinstance(out['relative_intensity'], float), f'Clip {idx}: relative_intensity is not float'
    assert 0.0 <= out['relative_intensity'] <= 1.0, f'Clip {idx}: intensity {out["relative_intensity"]} out of [0, 1]'
    
    test_outputs.append(out)

elapsed = time.perf_counter() - t0
print(f'Evaluated {len(test_outputs)} fold-10 clips in {elapsed:.2f} s ({elapsed/len(test_outputs)*1000:.1f} ms/clip)')
print('Schema assertions passed: all 605 outputs have required keys, valid classes, and scores in [0, 1] ✓')

print('\nSample Predictions (first 3 fold-10 clips):')
for k in range(3):
    idx_k = test_idx[k]
    print(f'  Clip {labels.loc[idx_k, "path"]}:')
    print(f'    {test_outputs[k]}')

Evaluated 605 fold-10 clips in 38.71 s (64.0 ms/clip)
Schema assertions passed: all 605 outputs have required keys, valid classes, and scores in [0, 1] ✓

Sample Predictions (first 3 fold-10 clips):
  Clip data/raw/UrbanSound8K/audio/fold10/100648-1-0-0.wav:
    {'type': 'construction', 'confidence': 0.9973998810134733, 'relative_intensity': 0.5085095327501679}
  Clip data/raw/UrbanSound8K/audio/fold10/100648-1-1-0.wav:
    {'type': 'construction', 'confidence': 0.8556360423684863, 'relative_intensity': 0.41989048954841945}
  Clip data/raw/UrbanSound8K/audio/fold10/100648-1-2-0.wav:
    {'type': 'traffic_horn', 'confidence': 0.6275321982853197, 'relative_intensity': 0.5140486818974571}


In [3]:
# Cell 3 – verify consistency with precomputed features and intensity scores
audio_features = np.load(REPO_ROOT / 'data/processed/audio_features.npy')
audio_intensity = np.load(REPO_ROOT / 'data/processed/audio_intensity.npy')
model = joblib.load(REPO_ROOT / 'models/svm_mixed_candidate.joblib')

X_test = audio_features[test_idx]
expected_types = model.predict(X_test)
expected_intensity = audio_intensity[test_idx]

actual_types = np.array([out['type'] for out in test_outputs])
actual_intensity = np.array([out['relative_intensity'] for out in test_outputs])

mismatches = int(np.sum(actual_types != expected_types))
print(f'Type mismatches vs model.predict on audio_features.npy[test_idx]: {mismatches} (expect 0)')
assert mismatches == 0, f'Assertion failed: found {mismatches} type mismatches!'

intensity_diffs = np.abs(actual_intensity - expected_intensity)
max_diff = float(np.max(intensity_diffs))
print(f'Max absolute intensity difference vs audio_intensity.npy[test_idx]: {max_diff:.2e} (expect <= 1e-9)')
assert max_diff <= 1e-9, f'Assertion failed: max intensity difference {max_diff} > 1e-9!'

print('Cell 3 asserts passed: 0 type mismatches and exact relative intensity agreement ✓')

Type mismatches vs model.predict on audio_features.npy[test_idx]: 0 (expect 0)
Max absolute intensity difference vs audio_intensity.npy[test_idx]: 0.00e+00 (expect <= 1e-9)
Cell 3 asserts passed: 0 type mismatches and exact relative intensity agreement ✓


In [4]:
# Cell 4 – fold-10 accuracy and predicted class distribution
y_test = labels.loc[test_idx, 'app_class'].to_numpy()
acc = float(accuracy_score(y_test, actual_types))

print(f'predict_audio Fold 10 Accuracy: {acc:.4f} ({acc*100:.2f}%)')
assert round(acc, 4) == 0.7818, f'Expected accuracy 0.7818 (SVM-mixed clean), got {acc:.4f}'
print('Accuracy assertion passed: matches 0.7818 to 4 decimal places ✓')

counts = pd.Series(actual_types).value_counts().sort_index()
print('\nPer-Class Predicted Counts (Fold 10, 605 clips):')
for cls, count in counts.items():
    print(f'  {cls:20s}: {count:3d} ({count/len(actual_types)*100:5.1f}%)')

predict_audio Fold 10 Accuracy: 0.7818 (78.18%)
Accuracy assertion passed: matches 0.7818 to 4 decimal places ✓

Per-Class Predicted Counts (Fold 10, 605 clips):
  construction        : 202 ( 33.4%)
  hvac                : 117 ( 19.3%)
  music_loudspeaker   : 110 ( 18.2%)
  siren               :  64 ( 10.6%)
  traffic_horn        : 112 ( 18.5%)


In [5]:
# Cell 5 – error handling and execution latency benchmark
missing_file = REPO_ROOT / 'data/non_existent_audio_file.wav'
print(f'Testing error handling on missing file: {missing_file}')

error_caught = False
try:
    predict_audio(missing_file)
except ValueError as e:
    error_caught = True
    print(f'Correctly raised ValueError: {e}')

assert error_caught, 'Assertion failed: predict_audio did not raise ValueError for a missing file!'
print('Error handling assertion passed ✓')

# Benchmark latency over 50 calls
print('\nBenchmarking latency over 50 calls...')
BENCHMARK_N = 50
t0 = time.perf_counter()
for idx in test_idx[:BENCHMARK_N]:
    path = REPO_ROOT / labels.loc[idx, 'path']
    _ = predict_audio(path)
bench_elapsed = time.perf_counter() - t0

mean_ms = (bench_elapsed / BENCHMARK_N) * 1000.0
print(f'Completed {BENCHMARK_N} calls in {bench_elapsed:.3f} s')
print(f'Mean latency per call: {mean_ms:.2f} ms')

Testing error handling on missing file: C:\Users\LENOVO\Desktop\CityPulse\data\non_existent_audio_file.wav
Correctly raised ValueError: Audio file does not exist or is not a valid file: C:\Users\LENOVO\Desktop\CityPulse\data\non_existent_audio_file.wav
Error handling assertion passed ✓

Benchmarking latency over 50 calls...


Completed 50 calls in 2.024 s
Mean latency per call: 40.48 ms


In [6]:
# Cell 6 – export model to models/audio_model.joblib and verify SHA256 equality
src_model_path = REPO_ROOT / 'models/svm_mixed_candidate.joblib'
dst_model_path = REPO_ROOT / 'models/audio_model.joblib'

shutil.copyfile(src_model_path, dst_model_path)
assert dst_model_path.exists(), f'Destination file {dst_model_path} was not created!'

audio_model_sha = compute_sha256(dst_model_path)
expected_sha = compute_sha256(src_model_path)

print(f'Source model SHA256     : {expected_sha}')
print(f'Exported audio_model SHA: {audio_model_sha}')

assert audio_model_sha == expected_sha, f'Hash mismatch: {audio_model_sha} != {expected_sha}'
print('Assertion passed: models/audio_model.joblib matches svm_mixed_candidate.joblib hash ✓')
print('\nCell 6 completed ✓')

Source model SHA256     : 72543a986a3178f952f7a935ab7054ad44cfc1e52dbcae30a2756fee95dbb03d
Exported audio_model SHA: 72543a986a3178f952f7a935ab7054ad44cfc1e52dbcae30a2756fee95dbb03d
Assertion passed: models/audio_model.joblib matches svm_mixed_candidate.joblib hash ✓

Cell 6 completed ✓
